# iNDI panel 2 metadata processing
The iNDI metadata is pretty broken up, I'm trying to unify into a cleaner metadata sheet for downstream analysis, this should be cleaned up a lot better down the road and processed my thuroughly from the original metadata sheets though

In [2]:
import pathlib as Path
import pandas as pd

## Set up paths

In [9]:
META_DIR = "/data/CARDPB2/iNDI/Production/metadata"

AUTO_FN = f"{META_DIR}/AbPanel2/AbPanel2_automated_plates_combined.csv"

JAX_FN  = f"{META_DIR}/JAX_iPSC_Catalog.tsv"
PLATE_ID_FN = f"{META_DIR}/indi_plateID_to_folderID.csv"
ASP_POS_ID_FN = f"{META_DIR}/asp_pos_ids_metadata.csv"
BATCH_FN = f"{META_DIR}/AbPanel2/batches.csv"

print(JAX_FN, PLATE_ID_FN, ASP_POS_ID_FN, BATCH_FN, sep="\n")

/data/CARDPB2/iNDI/Production/metadata/JAX_iPSC_Catalog.tsv
/data/CARDPB2/iNDI/Production/metadata/indi_plateID_to_folderID.csv
/data/CARDPB2/iNDI/Production/metadata/asp_pos_ids_metadata.csv
/data/CARDPB2/iNDI/Production/metadata/AbPanel2/batches.csv


## Process automated metadata

In [ ]:
AUTO = pd.read_csv(AUTO_FN)

AUTO["JAX_ID"] = AUTO["JAX_ID"].replace({"JIPSC1000": "JIPSC001000"})
AUTO["JAX_ID"] = AUTO["JAX_ID"].str.replace("JAX_ID_", "", regex=False)

print(AUTO.head)
print(AUTO.columns.to_list())

<bound method NDFrame.head of                                            Cell_Line_ID iNDI_Plate_ID Well_ID  \
0                    JAX_ID_JIPSC001002\nFUS R495X HOM     INDI00004D     A01   
1     JAX_ID_JIPSC001072\nVCP R159H REV \nLow counts...    INDI00004D     A02   
2                 JAX_ID_JIPSC001000\nParental  ID:5424    INDI00004D     A03   
3                    JAX_ID_JIPSC001006\nFUS R495X REV     INDI00004D     A04   
4                     JAX_ID_JIPSC001070\nVCP R159H HET    INDI00004D     A05   
...                                                 ...           ...     ...   
4603  JAX_ID_JIPSC001114\nUBQLN2 P506T REV\nLow coun...    INDI00054D     P20   
4604    JAX_ID_JIPSC001150\nAPOE C112R "e4" (A) JAX HOM    INDI00054D     P21   
4605  JAX_ID_JIPSC001128\nSNCA A53T HET\nLow counts ...    INDI00054D     P22   
4606                  JAX_ID_JIPSC001130\nSNCA A53T HOM    INDI00054D     P23   
4607         JAX_ID_JIPSC001000\nParental (P5) ID:10080    INDI00054D     P24  

## Get JAX IDs

In [18]:
## Get JAX IDs
JAX = pd.read_csv(JAX_FN, sep="\t")

AUTO = AUTO.merge(
    JAX[["Gene", "Variant", "Genotype", "JAX_Code"]],
    left_on=["JAX_ID"],
    right_on=["JAX_Code"],
    how="left",
)

AUTO = AUTO.rename(columns={"JAX_Code": "JAX_ID"})

print(AUTO.head())
print(AUTO.columns.to_list())

                                        Cell_Line_ID iNDI_Plate_ID Well_ID  \
0                 JAX_ID_JIPSC001002\nFUS R495X HOM     INDI00004D     A01   
1  JAX_ID_JIPSC001072\nVCP R159H REV \nLow counts...    INDI00004D     A02   
2              JAX_ID_JIPSC001000\nParental  ID:5424    INDI00004D     A03   
3                 JAX_ID_JIPSC001006\nFUS R495X REV     INDI00004D     A04   
4                  JAX_ID_JIPSC001070\nVCP R159H HET    INDI00004D     A05   

   Genotype_Name       JAX_ID Batch_Info      Gene Variant Genotype  \
0  FUS R495X HOM  JIPSC001002        NaN       FUS   R495X  SNV/SNV   
1  VCP R159H REV  JIPSC001072        NaN       VCP   R159H   REV/WT   
2       Parental  JIPSC001000    ID:5424  KOLF2.1J     NaN      NaN   
3  FUS R495X REV  JIPSC001006        NaN       FUS   R495X   REV/WT   
4  VCP R159H HET  JIPSC001070        NaN       VCP   R159H   SNV/WT   

        JAX_ID  
0  JIPSC001002  
1  JIPSC001072  
2  JIPSC001000  
3  JIPSC001006  
4  JIPSC001070  
['

## Process Asp PosID sheet

In [19]:
ASP_POS_ID = pd.read_csv(ASP_POS_ID_FN)
ASP_POS_ID = ASP_POS_ID.drop(columns=["Disp PosID", "Disp_Pos_ID"])

print(ASP_POS_ID.head())

COMBINED_ASP = AUTO.merge(
    ASP_POS_ID,
    left_on=["iNDI_Plate_ID", "Well_ID"],
    right_on=["iNDI_Plate_ID", "Well_ID"],
    how="left",
)

print(COMBINED_ASP.head())
print(COMBINED_ASP.columns.to_list())

  Asp PosID iNDI_Plate_ID Well_ID
0        A1    INDI00001D     D01
1        A1    INDI00001D     H01
2        A1    INDI00002D     D01
3        A1    INDI00002D     H01
4        A1    INDI00003D     D01
                                        Cell_Line_ID iNDI_Plate_ID Well_ID  \
0                 JAX_ID_JIPSC001002\nFUS R495X HOM     INDI00004D     A01   
1  JAX_ID_JIPSC001072\nVCP R159H REV \nLow counts...    INDI00004D     A02   
2              JAX_ID_JIPSC001000\nParental  ID:5424    INDI00004D     A03   
3                 JAX_ID_JIPSC001006\nFUS R495X REV     INDI00004D     A04   
4                  JAX_ID_JIPSC001070\nVCP R159H HET    INDI00004D     A05   

   Genotype_Name       JAX_ID Batch_Info      Gene Variant Genotype  \
0  FUS R495X HOM  JIPSC001002        NaN       FUS   R495X  SNV/SNV   
1  VCP R159H REV  JIPSC001072        NaN       VCP   R159H   REV/WT   
2       Parental  JIPSC001000    ID:5424  KOLF2.1J     NaN      NaN   
3  FUS R495X REV  JIPSC001006        NaN   

## Process batch sheet

In [20]:
BATCH = pd.read_csv(BATCH_FN)
print(BATCH.columns.to_list())

COMBINED_ASP["AbPanel"] = 2

COMBINED_ASP_BATCH = COMBINED_ASP.merge(
    BATCH,
    left_on=["iNDI_Plate_ID", "AbPanel"],
    right_on=["iNDI_Plate_ID", "AbPanel"],
    how="left",
)
print(COMBINED_ASP_BATCH.head)

['iNDI_Plate_ID', 'AbPanel', 'Batch']
<bound method NDFrame.head of                                            Cell_Line_ID iNDI_Plate_ID Well_ID  \
0                    JAX_ID_JIPSC001002\nFUS R495X HOM     INDI00004D     A01   
1     JAX_ID_JIPSC001072\nVCP R159H REV \nLow counts...    INDI00004D     A02   
2                 JAX_ID_JIPSC001000\nParental  ID:5424    INDI00004D     A03   
3                    JAX_ID_JIPSC001006\nFUS R495X REV     INDI00004D     A04   
4                     JAX_ID_JIPSC001070\nVCP R159H HET    INDI00004D     A05   
...                                                 ...           ...     ...   
4603  JAX_ID_JIPSC001114\nUBQLN2 P506T REV\nLow coun...    INDI00054D     P20   
4604    JAX_ID_JIPSC001150\nAPOE C112R "e4" (A) JAX HOM    INDI00054D     P21   
4605  JAX_ID_JIPSC001128\nSNCA A53T HET\nLow counts ...    INDI00054D     P22   
4606                  JAX_ID_JIPSC001130\nSNCA A53T HOM    INDI00054D     P23   
4607         JAX_ID_JIPSC001000\nParental

## Process plate ID sheet

In [21]:
PLATE_ID = pd.read_csv(PLATE_ID_FN)

print(PLATE_ID.head)
print(PLATE_ID.columns.to_list())

COMBINED_ASP_BATCH_ID = COMBINED_ASP_BATCH.merge(
    PLATE_ID,
    left_on=["iNDI_Plate_ID"],
    right_on=["iNDI_Plate_ID"],
    how="left",
)
print(COMBINED_ASP_BATCH_ID.head)

<bound method NDFrame.head of    iNDI_Plate_ID         Layout                             Folder_ID
0     INDI00002D       Plate002  2b1c31ad-579d-4ea6-a782-251ea083ed6a
1     INDI00007D       Plate007  d7e04b5c-a253-42ff-859f-2af0e6047a00
2     INDI00011D       Plate011  f8062781-891b-4688-a59e-fccbf48325cf
3     INDI00015D       Plate015  caef83c7-6b44-4b64-b8ec-e8d84ca6759d
4     INDI00019D       Plate019  c8f2dbbe-6632-4063-ab68-0b4fa37909af
5     INDI00023D       Plate023  32d75d22-bd17-4629-8257-a4056e3ffcee
6     INDI00026D       Plate026  c6a92be6-a13a-4957-a62d-9f99961c903f
7     INDI00035D       Plate002  c170959e-633a-458c-8542-2aa619c0d7f8
8     INDI00039D       Plate039  07fc5da6-9d7d-4c97-858b-4b76df1859a5
9     INDI00043D       Plate043  4405a3b2-6b88-49b1-91f3-992e09ccbd16
10    INDI00047D       Plate047  72c0a713-b5d5-4cd4-a826-c556368ed1d1
11    INDI00049D       Plate049  7e03b502-ad64-4090-a9e2-fc4c3ecdae11
12    INDI00053D       Plate053  dad5d97b-1a66-4a2e-9ea2-bde

## Write final metadata sheet

In [22]:
COMBINED_ASP_BATCH_ID.to_csv("COMBINED_ASP_BATCH_METADATA.csv", index=False)